# Séance 6 · Collecter des données avec une API · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

Jusqu'ici, les données arrivaient dans un fichier CSV tout prêt. Mais la plupart des données du monde ne sont pas dans des fichiers : elles sont **derrière des sites**, et on les demande avec une **API**. Aujourd'hui, tu vas demander des fiches de Pokémon et des prévisions météo, lire leurs réponses en JSON, les ranger dans un tableau, et prévoir un plan B quand le réseau lâche.

Ce notebook tourne dans **Google Colab** : rien à installer. `Maj + Entrée` pour exécuter une cellule.

**Livrable de la séance** : ton propre jeu de données, collecté avec une API, rangé dans un tableau et sauvegardé en CSV.

## Préparation

`requests` sert à appeler les API (il est déjà dans Colab). La fonction `demander(url, secours, params)` appelle une API et renvoie sa réponse ; si le réseau ne répond pas, elle renvoie les données de **secours** pour que le notebook aille quand même au bout. On l'écrit ensemble à la section 6 : pour l'instant, on s'en sert.

In [ ]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt

def demander(url, secours, params=None):
    """Appelle l'API et renvoie son JSON ; si ça rate (pas de réseau, erreur), renvoie les données de secours."""
    try:
        reponse = requests.get(url, params=params, timeout=10)
        reponse.raise_for_status()
        return reponse.json()
    except Exception as erreur:
        print("Pas de réponse de l'API, on utilise les données de secours :", erreur)
        return secours

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok

SECOURS_PIKACHU = {"name": "pikachu", "height": 4, "weight": 60,
                   "types": [{"type": {"name": "electric"}}],
                   "stats": [{"stat": {"name": "hp"}, "base_stat": 35}, {"stat": {"name": "attack"}, "base_stat": 55},
                             {"stat": {"name": "defense"}, "base_stat": 40}, {"stat": {"name": "speed"}, "base_stat": 90}]}
SECOURS_METEO = {"hourly": {"time": [f"2026-10-17T{h:02d}:00" for h in range(24)],
                            "temperature_2m": [11, 10.6, 10.2, 9.9, 9.7, 9.5, 9.6, 10.1, 11.2, 12.6, 14, 15.2,
                                               16.1, 16.7, 17, 16.9, 16.4, 15.5, 14.4, 13.5, 12.8, 12.3, 11.9, 11.5]}}
print("Prêt !")

## 1. Une API, c'est quoi ?

Au restaurant, tu ne vas pas en cuisine : tu passes commande au serveur, avec des mots précis, et il revient avec le plat. Une **API** (*Application Programming Interface*), c'est ce serveur :

- la **commande**, c'est une **URL** : `https://pokeapi.co/api/v2/pokemon/pikachu` ;
- le serveur, c'est `requests.get(url)` ;
- le **plat**, c'est la réponse, presque toujours en **JSON**, un format qui ressemble à un dictionnaire Python.

Premier essai : la [PokéAPI](https://pokeapi.co), gratuite et sans inscription.

On passe commande. `requests.get` envoie l'URL et attend la réponse (10 secondes au plus).

In [ ]:
URL_PIKACHU = "https://pokeapi.co/api/v2/pokemon/pikachu"
try:
    reponse = requests.get(URL_PIKACHU, timeout=10)
except requests.RequestException as erreur:
    reponse = None
    print("Pas de réseau :", erreur)

Le **code de réponse** dit comment ça s'est passé : `200` = tout va bien, `404` = introuvable, `500` = le site a un problème.

In [ ]:
print(reponse.status_code if reponse is not None else "aucune réponse")

Le texte brut de la réponse : du JSON, c'est-à-dire du texte avec des accolades, des crochets et des guillemets.

In [ ]:
print(reponse.text[:300] if reponse is not None else "aucune réponse")

`.json()` transforme ce texte en vrai dictionnaire Python. À partir d'ici, on passe par `demander`, qui fait la même chose avec un plan B.

In [ ]:
pikachu = demander(URL_PIKACHU, SECOURS_PIKACHU)
print(type(pikachu))

Quelles **clés** y a-t-il dans ce dictionnaire ? Il y en a beaucoup : on regarde les 15 premières.

In [ ]:
print(list(pikachu.keys())[:15])

Une clé simple donne une valeur simple.

In [ ]:
print(pikachu["name"], "· taille :", pikachu["height"], "· poids :", pikachu["weight"])

## 2. Lire un JSON imbriqué

Le JSON est souvent **imbriqué** : des dictionnaires dans des listes, dans des dictionnaires. Les types de Pikachu sont une liste de dictionnaires, chacun avec un dictionnaire `type` à l'intérieur.

In [ ]:
pikachu["types"]

On descend marche par marche : le premier élément de la liste, sa clé `type`, puis sa clé `name`.

In [ ]:
pikachu["types"][0]["type"]["name"]

Pour tous les types d'un coup, une liste en compréhension parcourt la liste.

In [ ]:
types_pikachu = [t["type"]["name"] for t in pikachu["types"]]
types_pikachu

Les stats sont rangées pareil. On en fait un dictionnaire simple : le nom de la stat donne sa valeur.

In [ ]:
stats_pikachu = {s["stat"]["name"]: s["base_stat"] for s in pikachu["stats"]}
stats_pikachu

### À toi : la vitesse de Pikachu

Range dans `vitesse` la stat `speed` de Pikachu, en partant du dictionnaire `pikachu` (pas de `stats_pikachu`, pour t'entraîner à descendre dans le JSON).

In [ ]:
# À toi
vitesse = None

In [ ]:
verifier("La vitesse de Pikachu", vitesse == stats_pikachu["speed"])

<details><summary>Solution</summary>

```python
vitesse = [s["base_stat"] for s in pikachu["stats"] if s["stat"]["name"] == "speed"][0]
print(vitesse)
```

</details>

## 3. Plusieurs appels, un tableau

Un appel donne **une** fiche. Pour un tableau, on fait un appel par Pokémon, et chaque appel devient une ligne. On range ça dans une fonction.

In [ ]:
def fiche_pokemon(nom):
    """Une ligne de tableau pour un Pokémon : son nom, son premier type, sa taille, son poids et ses stats."""
    data = demander(f"https://pokeapi.co/api/v2/pokemon/{nom}", dict(SECOURS_PIKACHU, name=nom))
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    return {"nom": data["name"], "type": data["types"][0]["type"]["name"],
            "taille": data["height"], "poids": data["weight"], **stats}

On essaie la fonction sur un seul Pokémon : un dictionnaire, c'est une ligne.

In [ ]:
fiche_pokemon("snorlax")

On l'appelle pour cinq Pokémon, en laissant une petite pause entre deux appels : une API gratuite n'aime pas qu'on la bombarde.

In [ ]:
lignes = []
for nom in ["pikachu", "bulbasaur", "charmander", "squirtle", "snorlax"]:
    lignes.append(fiche_pokemon(nom))
    time.sleep(0.2)

Une liste de dictionnaires devient un tableau pandas en une ligne.

In [ ]:
equipe = pd.DataFrame(lignes)
equipe

On vérifie : cinq lignes, et des colonnes de nombres.

In [ ]:
equipe.dtypes

Et voilà des données collectées soi-même, prêtes pour un graphique.

In [ ]:
equipe.plot(kind="bar", x="nom", y="poids", legend=False, title="Le poids de mon équipe (en hectogrammes)")
plt.ylabel("poids (hg)")
plt.show()

## 4. Des paramètres dans la commande

Beaucoup d'API prennent des **paramètres** : la ville, la date, le nombre de résultats. Ils s'écrivent dans l'URL après un `?`, mais `requests` les écrit pour nous à partir d'un dictionnaire `params`.

Deuxième API : [Open-Meteo](https://open-meteo.com), la météo gratuite et sans clé. On lui donne une latitude, une longitude, et ce qu'on veut : la température heure par heure pour aujourd'hui.

In [ ]:
URL_METEO = "https://api.open-meteo.com/v1/forecast"
params_paris = {"latitude": 48.85, "longitude": 2.35, "hourly": "temperature_2m",
                "forecast_days": 1, "timezone": "Europe/Paris"}

`requests` construit l'URL complète à partir des paramètres : c'est elle que le site reçoit.

In [ ]:
requests.Request("GET", URL_METEO, params=params_paris).prepare().url

On passe commande.

In [ ]:
meteo = demander(URL_METEO, SECOURS_METEO, params=params_paris)

La réponse range les heures et les températures dans deux listes de même longueur, sous la clé `hourly`.

In [ ]:
print(meteo["hourly"]["time"][:3], meteo["hourly"]["temperature_2m"][:3])

Deux listes de même longueur font deux colonnes d'un tableau.

In [ ]:
heures = pd.DataFrame({"heure": pd.to_datetime(meteo["hourly"]["time"]),
                       "temperature": meteo["hourly"]["temperature_2m"]})
heures.head()

La journée de Paris, en une courbe.

In [ ]:
plt.plot(heures["heure"].dt.hour, heures["temperature"], marker="o")
plt.xlabel("heure"); plt.ylabel("température (°C)"); plt.title("Aujourd'hui à Paris, heure par heure")
plt.show()

### À toi : trois villes

Compare la température **maximale** d'aujourd'hui dans trois villes. Construis `max_villes`, un tableau avec une ligne par ville (colonnes `ville` et `temperature_max`).
Coordonnées : Paris (48.85, 2.35), Marseille (43.30, 5.37), Lille (50.63, 3.06). Les paramètres sont ceux de Paris, avec une autre latitude et une autre longitude.

In [ ]:
# À toi
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}
max_villes = None

In [ ]:
verifier("Une ligne par ville", max_villes is not None and len(max_villes) == 3 and set(max_villes["ville"]) == set(villes))
verifier("Des températures", max_villes is not None and max_villes["temperature_max"].notna().all())

<details><summary>Solution</summary>

```python
villes = {"Paris": (48.85, 2.35), "Marseille": (43.30, 5.37), "Lille": (50.63, 3.06)}
lignes = []
for ville, (lat, lon) in villes.items():
    params = dict(params_paris, latitude=lat, longitude=lon)
    temperatures = demander(URL_METEO, SECOURS_METEO, params=params)["hourly"]["temperature_2m"]
    lignes.append({"ville": ville, "temperature_max": max(temperatures)})
    time.sleep(0.2)
max_villes = pd.DataFrame(lignes).sort_values("temperature_max", ascending=False)
max_villes
```

</details>

## 5. Quand ça rate

Une API peut ne pas répondre : pas de réseau, site en panne, nom mal écrit. On provoque une erreur exprès, avec un Pokémon qui n'existe pas.

In [ ]:
URL_FAUSSE = "https://pokeapi.co/api/v2/pokemon/pikachou"
try:
    echec = requests.get(URL_FAUSSE, timeout=10)
except requests.RequestException as erreur:
    echec = None

Le code dit « introuvable ». La réponse n'est pas du JSON : `.json()` planterait.

In [ ]:
print(echec.status_code if echec is not None else "aucune réponse")

`raise_for_status()` transforme un code d'erreur en vraie erreur Python : c'est ce qui permet à `demander` de basculer sur le plan B.

In [ ]:
try:
    echec.raise_for_status()
except Exception as erreur:
    print("Erreur repérée :", erreur)

Avec `demander`, la même faute de frappe ne fait pas planter le notebook : on récupère les données de secours, et on le sait.

In [ ]:
secours = demander(URL_FAUSSE, {"name": "inconnu"})
secours["name"]

## 6. Écrire son propre `demander`

Les trois idées de cette séance tiennent dans une fonction : appeler avec des paramètres, refuser un code d'erreur, et avoir un plan B. Voici `demander`, ligne par ligne :

- `requests.get(url, params=params, timeout=10)` passe la commande, et n'attend pas plus de 10 secondes ;
- `raise_for_status()` lève une erreur si le code n'est pas un succès ;
- `return reponse.json()` renvoie le dictionnaire ;
- `except Exception` attrape tout ce qui a echec, et `return secours` rend le plan B.

On vérifie que les deux chemins marchent : la vraie réponse, puis le secours.

In [ ]:
print(demander(URL_PIKACHU, {"name": "secours"})["name"])
print(demander(URL_FAUSSE, {"name": "secours"})["name"])

## 7. Garder ce qu'on a collecté

Appeler l'API à chaque fois qu'on rouvre le notebook, c'est lent, et ça fatigue le site. On sauvegarde le tableau en CSV, une fois pour toutes.

In [ ]:
equipe.to_csv("mon_equipe.csv", index=False)

On le recharge comme n'importe quel fichier, sans plus toucher à l'API.

In [ ]:
relu = pd.read_csv("mon_equipe.csv")
relu.shape

On vérifie que rien ne s'est perdu en route : mêmes lignes, mêmes colonnes.

In [ ]:
verifier("Le CSV contient toute l'équipe", relu.shape == equipe.shape and list(relu.columns) == list(equipe.columns))

## 8. Projet : ton propre jeu de données

Collecte un petit jeu de données **à toi** avec une API, au choix :

- **ton équipe de 6 Pokémon** avec la PokéAPI (colonnes : nom, type, poids, au moins deux stats) ;
- **la météo de 5 villes** qui comptent pour toi avec Open-Meteo (une ligne par ville : température max, min, pluie) ;
- **une autre API gratuite et sans clé** de la liste https://github.com/public-apis/public-apis (colonne *Auth* = `No`).

Consignes :
1. au moins 5 lignes et 3 colonnes, construites avec une boucle et `demander` ;
2. une pause entre deux appels (`time.sleep(0.2)`) ;
3. un graphique qui répond à une question sur tes données ;
4. le tableau sauvegardé en CSV : c'est ton livrable.

In [ ]:
# À toi : la collecte (une boucle, demander, une liste de dictionnaires)

In [ ]:
# À toi : le tableau, puis son graphique

In [ ]:
# À toi : la sauvegarde en CSV

## À retenir

- Une **API** = une URL qu'on demande avec `requests.get`, et une réponse en **JSON**.
- Le **code de réponse** dit si ça a marché : 200 = oui, 404 = introuvable.
- Un JSON imbriqué se lit marche par marche : `data["types"][0]["type"]["name"]`.
- Plusieurs appels → une liste de dictionnaires → `pd.DataFrame`.
- Les **paramètres** (`params=`) précisent la commande ; une **pause** entre deux appels ménage le site.
- Toujours un **plan B** (`try / except`, des données de secours), et un CSV pour ne pas tout redemander.

## Pour montrer aux autres

1. Quelle API as-tu utilisée, et qu'as-tu collecté (combien de lignes, quelles colonnes) ?
2. Montre ton graphique : à quelle question répond-il ?
3. Qu'est-ce qui a echec pendant la collecte, et comment ton plan B l'a-t-il rattrapé ?

Liens gratuits
- PokéAPI, la documentation : https://pokeapi.co/docs/v2
- Open-Meteo, la documentation : https://open-meteo.com/en/docs
- Liste d'API publiques gratuites : https://github.com/public-apis/public-apis